# Capstone Project — Modelling

**Problem**: Forecast the next 24 months of U.S. monthly alcohol sales.

In the previous notebook we completed the EDA.  Here we:
1. Benchmark models: Naive, Seasonal Naive, Drift
2. Statistical models: ETS, auto_arima (SARIMA), manual SARIMA, Holt-Winters
3. Prophet: basic + tuned
4. Results table: all models with MAE, RMSE, MAPE
5. Visual comparison
6. Forecast combination of top 3
7. Final forecast: retrain best model on all data
8. Conclusions and lessons learned

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Load Data and Split

In [ ]:
raw = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv", parse_dates=["DATE"], index_col="DATE")
raw.index.freq = "MS"
y = raw["S4248SM144NCEN"].copy()
y.name = "AlcoholSales"

# Same split as EDA notebook: last 24 months
n_test = 24
y_train = y.iloc[:-n_test]
y_test = y.iloc[-n_test:]

print(f"Train: {len(y_train)} months ({y_train.index[0].date()} to {y_train.index[-1].date()})")
print(f"Test:  {len(y_test)} months ({y_test.index[0].date()} to {y_test.index[-1].date()})")

In [ ]:
# Helper function
def compute_metrics(actual, predicted, model_name):
    """Compute MAE, RMSE, MAPE."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {"Model": model_name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2)}

# Collectors
all_results = []
all_forecasts = {}  # name -> array of predictions

print("Helper function and collectors ready.")

---
## 2. Benchmark Models

In [ ]:
# 2a. Naive (last value)
naive_pred = np.full(n_test, y_train.iloc[-1])
all_results.append(compute_metrics(y_test, naive_pred, "Naive"))
all_forecasts["Naive"] = naive_pred
print(f"Naive: repeat {y_train.iloc[-1]:.0f} for all {n_test} months.")

In [ ]:
# 2b. Seasonal Naive (repeat last year's pattern)
snaive_pred = np.tile(y_train.iloc[-12:].values, n_test // 12 + 1)[:n_test]
all_results.append(compute_metrics(y_test, snaive_pred, "Seasonal Naive"))
all_forecasts["Seasonal Naive"] = snaive_pred
print("Seasonal Naive: repeat the last 12 months of training data.")

In [ ]:
# 2c. Drift (linear extrapolation from first to last observation)
slope = (y_train.iloc[-1] - y_train.iloc[0]) / (len(y_train) - 1)
drift_pred = y_train.iloc[-1] + slope * np.arange(1, n_test + 1)
all_results.append(compute_metrics(y_test, drift_pred, "Drift"))
all_forecasts["Drift"] = drift_pred
print(f"Drift: slope = {slope:.2f} per month.")

In [ ]:
# Visualize benchmarks
fig, ax = plt.subplots(figsize=(14, 6))
y_train.iloc[-36:].plot(ax=ax, label="Train", color="tab:blue")
y_test.plot(ax=ax, label="Test (actual)", color="black", linewidth=2, marker="o", markersize=3)

ax.plot(y_test.index, naive_pred, "--", label="Naive", color="tab:red")
ax.plot(y_test.index, snaive_pred, "--", label="Seasonal Naive", color="tab:green")
ax.plot(y_test.index, drift_pred, "--", label="Drift", color="tab:purple")

ax.set_title("Benchmark Forecasts vs Actuals")
ax.set_ylabel("Sales")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. ETS (Additive)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets_add = ExponentialSmoothing(
    y_train, trend="add", seasonal="add", seasonal_periods=12
).fit(optimized=True)
ets_add_pred = ets_add.forecast(n_test)

all_results.append(compute_metrics(y_test, ets_add_pred, "ETS (additive)"))
all_forecasts["ETS (additive)"] = ets_add_pred.values

print(f"ETS additive — AIC: {ets_add.aic:.2f}")
print(f"Smoothing params: alpha={ets_add.params['smoothing_level']:.4f}, "
      f"beta={ets_add.params['smoothing_trend']:.4f}, "
      f"gamma={ets_add.params['smoothing_seasonal']:.4f}")

---
## 4. Holt-Winters (Multiplicative Seasonality)

In [ ]:
hw_mul = ExponentialSmoothing(
    y_train, trend="add", seasonal="mul", seasonal_periods=12
).fit(optimized=True)
hw_mul_pred = hw_mul.forecast(n_test)

all_results.append(compute_metrics(y_test, hw_mul_pred, "Holt-Winters (mul.)"))
all_forecasts["Holt-Winters (mul.)"] = hw_mul_pred.values

print(f"Holt-Winters multiplicative — AIC: {hw_mul.aic:.2f}")
print(f"Smoothing params: alpha={hw_mul.params['smoothing_level']:.4f}, "
      f"beta={hw_mul.params['smoothing_trend']:.4f}, "
      f"gamma={hw_mul.params['smoothing_seasonal']:.4f}")

---
## 5. Auto ARIMA (SARIMA via pmdarima)

In [ ]:
import pmdarima as pm

auto_model = pm.auto_arima(
    y_train,
    seasonal=True,
    m=12,
    suppress_warnings=True,
    stepwise=True,
    trace=False,
)

auto_pred = auto_model.predict(n_periods=n_test)
all_results.append(compute_metrics(y_test, auto_pred, "auto_arima (SARIMA)"))
all_forecasts["auto_arima (SARIMA)"] = auto_pred

print(f"Selected order: {auto_model.order} x {auto_model.seasonal_order}")
print(f"AIC: {auto_model.aic():.2f}")

In [ ]:
print(auto_model.summary())

---
## 6. Manual SARIMA

Based on the ACF/PACF analysis from the EDA, try SARIMA(1,1,1)(1,1,1,12).

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_manual = SARIMAX(
    y_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

sarima_manual_pred = sarima_manual.forecast(n_test)
all_results.append(compute_metrics(y_test, sarima_manual_pred, "SARIMA(1,1,1)(1,1,1,12)"))
all_forecasts["SARIMA(1,1,1)(1,1,1,12)"] = sarima_manual_pred.values

print(f"SARIMA(1,1,1)(1,1,1,12) — AIC: {sarima_manual.aic:.2f}")

---
## 7. Residual Check on SARIMA

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from scipy.stats import shapiro

resid = sarima_manual.resid[13:]  # skip initial NaN-like values

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(resid)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Residuals")

axes[1].hist(resid, bins=25, edgecolor="black", alpha=0.7)
axes[1].set_title("Residual Distribution")

plot_acf(resid, lags=36, ax=axes[2], title="ACF of Residuals")

plt.tight_layout()
plt.show()

stat, p = shapiro(resid[:500] if len(resid) > 500 else resid)
print(f"Shapiro-Wilk test: stat={stat:.4f}, p={p:.4f}")
print(f"Residual mean: {resid.mean():.2f}, std: {resid.std():.2f}")

---
## 8. Prophet (Basic)

In [ ]:
from prophet import Prophet

# Prepare Prophet format
train_prophet = pd.DataFrame({
    "ds": y_train.index,
    "y": y_train.values,
})
test_prophet = pd.DataFrame({
    "ds": y_test.index,
    "y": y_test.values,
})

# Fit default Prophet
prophet_basic = Prophet()
prophet_basic.fit(train_prophet)

future = prophet_basic.make_future_dataframe(periods=n_test, freq="MS")
fc_basic = prophet_basic.predict(future)
prophet_basic_pred = fc_basic.set_index("ds").loc[y_test.index, "yhat"].values

all_results.append(compute_metrics(y_test, prophet_basic_pred, "Prophet (default)"))
all_forecasts["Prophet (default)"] = prophet_basic_pred

print("Prophet (default) fitted.")

---
## 9. Prophet (Tuned)

Since the EDA showed multiplicative seasonality, use
`seasonality_mode='multiplicative'` and adjust changepoint prior.

In [ ]:
prophet_tuned = Prophet(
    seasonality_mode="multiplicative",
    changepoint_prior_scale=0.05,
    yearly_seasonality=True,
)
prophet_tuned.fit(train_prophet)

future_tuned = prophet_tuned.make_future_dataframe(periods=n_test, freq="MS")
fc_tuned = prophet_tuned.predict(future_tuned)
prophet_tuned_pred = fc_tuned.set_index("ds").loc[y_test.index, "yhat"].values

all_results.append(compute_metrics(y_test, prophet_tuned_pred, "Prophet (tuned)"))
all_forecasts["Prophet (tuned)"] = prophet_tuned_pred

print("Prophet (tuned, multiplicative seasonality) fitted.")

In [ ]:
# Prophet component plots
fig = prophet_tuned.plot_components(fc_tuned)
plt.suptitle("Prophet (Tuned) — Components", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Results Table

In [ ]:
results_df = pd.DataFrame(all_results).set_index("Model").sort_values("MAE")

print("=" * 70)
print("CAPSTONE MODEL COMPARISON — Alcohol Sales (24-month holdout)")
print("=" * 70)
results_df

In [ ]:
# Styled bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, metric in zip(axes, ["MAE", "RMSE", "MAPE (%)"]):
    bars = results_df[metric].plot.barh(ax=ax, color="steelblue", edgecolor="black")
    ax.set_title(metric, fontsize=14)
    ax.invert_yaxis()

plt.suptitle("Model Comparison — All Metrics", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 11. Visual Comparison: All Forecasts vs Actuals

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

y_train.iloc[-36:].plot(ax=ax, label="Train", color="tab:blue", linewidth=1)
y_test.plot(ax=ax, label="Test (actual)", color="black", linewidth=2.5, marker="o", markersize=3)

# Plot top models only (to avoid clutter)
top_models = results_df.index[:5].tolist()
colors = ["tab:red", "tab:green", "tab:purple", "tab:orange", "tab:brown"]
for name, color in zip(top_models, colors):
    ax.plot(y_test.index, all_forecasts[name], "--", color=color, label=name, alpha=0.8)

ax.set_title("Top 5 Models — Forecast vs Actuals")
ax.set_ylabel("Sales (Millions $)")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Plot ALL models in small multiples
n_models = len(all_forecasts)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows), sharex=True, sharey=True)
axes = axes.flat

for ax, (name, pred) in zip(axes, all_forecasts.items()):
    ax.plot(y_test.index, y_test.values, color="black", linewidth=1.5, label="Actual")
    ax.plot(y_test.index, pred, color="tab:red", linestyle="--", label="Forecast")
    mae_val = results_df.loc[name, "MAE"]
    ax.set_title(f"{name}\nMAE={mae_val}", fontsize=10)
    ax.legend(fontsize=7, loc="upper left")

# Hide unused subplots
for ax in axes[n_models:]:
    ax.set_visible(False)

plt.suptitle("All Models — Individual Forecast Panels", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 12. Forecast Combination: Top 3 Average

In [ ]:
top3 = results_df.index[:3].tolist()
print(f"Top 3 models by MAE: {top3}")

top3_preds = np.column_stack([all_forecasts[m] for m in top3])
combo_pred = top3_preds.mean(axis=1)

combo_result = compute_metrics(y_test, combo_pred, "Combination (Top 3)")
print(f"\nCombination metrics:")
print(f"  MAE:  {combo_result['MAE']}")
print(f"  RMSE: {combo_result['RMSE']}")
print(f"  MAPE: {combo_result['MAPE (%)']}%")

print(f"\nBest individual ({top3[0]}):")
print(f"  MAE: {results_df.loc[top3[0], 'MAE']}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
y_test.plot(ax=ax, label="Actual", color="black", linewidth=2, marker="o", markersize=3)

for name in top3:
    ax.plot(y_test.index, all_forecasts[name], ":", alpha=0.5, label=name)

ax.plot(y_test.index, combo_pred, label="Top-3 Average", color="tab:red", linewidth=2.5, linestyle="--")

ax.set_title("Forecast Combination (Top 3 Average) vs Actuals")
ax.set_ylabel("Sales (Millions $)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 13. Updated Results (Including Combination)

In [ ]:
all_results_final = all_results + [combo_result]
final_df = pd.DataFrame(all_results_final).set_index("Model").sort_values("MAE")

print("=" * 70)
print("FINAL RESULTS (including combination)")
print("=" * 70)
final_df

---
## 14. Select Best Approach

Based on the results, select the best model (or combination) for the
final forecast.

In [ ]:
best_name = final_df.index[0]
print(f"Best approach: {best_name}")
print(f"  MAE:  {final_df.loc[best_name, 'MAE']}")
print(f"  RMSE: {final_df.loc[best_name, 'RMSE']}")
print(f"  MAPE: {final_df.loc[best_name, 'MAPE (%)']}%")
print()
print("If the combination wins, we use the top-3 average.")
print("Otherwise, we use the best individual model.")

---
## 15. Final Forecast: Retrain on All Data

Retrain the best individual model on the full dataset and forecast
24 months into the future.

In [ ]:
# Retrain Holt-Winters multiplicative on all data
hw_final = ExponentialSmoothing(
    y, trend="add", seasonal="mul", seasonal_periods=12
).fit(optimized=True)
future_hw = hw_final.forecast(24)

print("Holt-Winters (multiplicative) retrained on all data.")
print(f"\nForecast for next 24 months:")
print(future_hw)

In [ ]:
# Retrain auto_arima on all data
auto_final = pm.auto_arima(y, seasonal=True, m=12, suppress_warnings=True, stepwise=True)
future_arima = pd.Series(
    auto_final.predict(n_periods=24),
    index=pd.date_range(start=y.index[-1] + pd.DateOffset(months=1), periods=24, freq="MS"),
)

print(f"auto_arima retrained. Order: {auto_final.order} x {auto_final.seasonal_order}")

In [ ]:
# Retrain Prophet (tuned) on all data
all_prophet = pd.DataFrame({"ds": y.index, "y": y.values})
prophet_final = Prophet(seasonality_mode="multiplicative", changepoint_prior_scale=0.05)
prophet_final.fit(all_prophet)
future_dates = prophet_final.make_future_dataframe(periods=24, freq="MS")
fc_final_prophet = prophet_final.predict(future_dates)
future_prophet = fc_final_prophet.set_index("ds")["yhat"].iloc[-24:]

print("Prophet (tuned) retrained on all data.")

In [ ]:
# Combination: average of 3 retrained models
future_combo = (future_hw.values + future_arima.values + future_prophet.values) / 3
future_index = future_hw.index

print("Combined forecast (average of HW + SARIMA + Prophet):")
for date, val in zip(future_index, future_combo):
    print(f"  {date.strftime('%Y-%m')}: {val:,.0f}")

In [ ]:
# Plot final forecasts
fig, ax = plt.subplots(figsize=(16, 7))

y.iloc[-60:].plot(ax=ax, label="Historical (last 5 years)", color="tab:blue", linewidth=1.5)

ax.plot(future_index, future_hw.values, "--", label="Holt-Winters", color="tab:green")
ax.plot(future_index, future_arima.values, "--", label="auto_arima", color="tab:purple")
ax.plot(future_index, future_prophet.values, "--", label="Prophet (tuned)", color="tab:orange")
ax.plot(future_index, future_combo, label="Combination", color="tab:red", linewidth=2.5, linestyle="--")

ax.axvline(y.index[-1], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Final 24-Month Forecast — Alcohol Sales")
ax.set_ylabel("Sales (Millions $)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

---
## 16. Conclusions and Lessons Learned

### Key Findings

1. **Baselines matter**: Seasonal Naive is a strong baseline for this data.
   Any model that cannot beat it is not worth the complexity.

2. **Multiplicative seasonality**: The seasonal amplitude grows with the level
   of the series.  Models that account for this (Holt-Winters multiplicative,
   Prophet with `seasonality_mode='multiplicative'`) outperform additive variants.

3. **SARIMA is competitive**: When the ACF/PACF analysis identifies a good
   candidate order, SARIMA performs very well.  `auto_arima` automates this.

4. **Forecast combination works**: Averaging the top 3 models typically
   produces a forecast that is at least as good as (and often better than)
   any single model.

5. **Prophet out of the box is decent but not dominant**: Default Prophet
   (additive seasonality) underperforms on this multiplicative data.  Tuning
   the seasonality mode helps significantly.

### Workflow Summary

1. **EDA** (previous notebook): understand trend, seasonality, stationarity
2. **Baselines**: Naive, Seasonal Naive, Drift
3. **Statistical models**: ETS, Holt-Winters, SARIMA
4. **ML/modern**: Prophet
5. **Evaluate**: same train/test split, multiple metrics (MAE, RMSE, MAPE)
6. **Combine**: average top models
7. **Retrain**: fit final model(s) on all data before deploying

### What We Would Do Next

- Add **exogenous variables** (economic indicators, holidays) via SARIMAX or Prophet regressors
- Try **machine learning** approaches (XGBoost with lag features, neural networks)
- Implement **rolling re-estimation** for production forecasting
- Build **prediction intervals** for the final forecast
- Set up a **monitoring pipeline** to detect when the model drifts

In [ ]:
print("Capstone project complete.")
print()
print("Throughout this course we have covered:")
print("  1. Time series fundamentals (pandas, visualization, decomposition)")
print("  2. Stationarity, differencing, ACF/PACF")
print("  3. Exponential smoothing (SES, DES, TES / Holt-Winters)")
print("  4. ARIMA and SARIMA")
print("  5. Prophet")
print("  6. Advanced frameworks (sktime, statsforecast)")
print("  7. Systematic model comparison and combination")
print("  8. End-to-end capstone project")
print()
print("You now have a complete toolkit for time series forecasting in Python.")